# 322. Coin Change
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/coin-change/

## 💡 Concepts

**Core concept(s):** Bottom-up DP over amounts — the fewest coins for amount `a` = 1 + the best of `a - coin` over all coins.

**Why it applies here:** Every amount's answer is built from smaller amounts you've already solved. Trying coins greedily fails (e.g. coins 1,3,4 for 6). DP tries all coins for each amount and reuses the smaller answers, so it's both correct and fast.

**Key intuition:** Fewest coins for `a` = 1 + the cheapest way to make `a - coin`, over every coin.

---

### 📚 What is Dynamic Programming (DP)?
**DP** solves a big problem by solving smaller **overlapping** subproblems once and reusing the answers. Two styles: **memoization** (recursion that caches results) and **tabulation** (fill a table from the smallest cases up).
- **Why it's fast:** it turns exponential re-computation into a single sweep over the subproblems.
- **In Python:** a `dict`/list cache, or a `dp` list/2-D table.

### 📚 Subproblems & Recurrence
The heart of DP is a **recurrence**: the answer for a state written in terms of smaller states (e.g. `dp[i] = dp[i-1] + dp[i-2]`). Find the recurrence and the base cases, and the code writes itself.

---

**Prerequisite knowledge:**
- Building answers from smaller amounts.
- Handling 'impossible' with infinity.

## 📝 Problem

Given coin denominations and a target `amount`, return the fewest coins that sum to it, or `-1` if impossible. Unlimited coins.

**Example**
```
coins = [1,2,5], amount = 11 -> 3   (5 + 5 + 1)
coins = [2], amount = 3 -> -1
```

> Two approaches: exponential brute force and O(amount × coins) DP.

### Approach 1 — Brute Recursion (worst)

**Idea:** Try subtracting each coin and recurse on the remainder; take the minimum. Recomputes remainders repeatedly.

**Time:** exponential. **Space:** `O(amount)`.

In [ ]:
def coin_change_brute(coins, amount):
    INF = float("inf")
    def dfs(rem):                          # fewest coins to make the remaining amount
        if rem == 0:
            return 0                       # nothing left to make
        if rem < 0:
            return INF                     # overshot -> this path is impossible
        best = INF
        for c in coins:                    # try using each coin once here...
            best = min(best, dfs(rem - c) + 1)  # ...then solve the smaller remainder
        return best
    res = dfs(amount)
    return res if res != INF else -1

### Approach 2 — Bottom-Up DP (optimal)

**Idea:** `dp[a]` = fewest coins for amount `a`. For each amount, try every coin and reuse `dp[a-c]`.

**Time:** `O(amount × coins)`. **Space:** `O(amount)`.

In [ ]:
def coin_change_dp(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount              # dp[a] = fewest coins to make amount a; dp[0]=0
    for a in range(1, amount + 1):         # build every amount from smaller amounts
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1       # use coin c on top of the best way to make (a-c)
    return dp[amount] if dp[amount] != INF else -1   # INF means "impossible"

In [ ]:
# Correctness check
tests = [([1,2,5],11,3), ([2],3,-1), ([1],0,0), ([1,3,4],6,2)]
for coins, amt, exp in tests:
    a, b = coin_change_brute(coins, amt), coin_change_dp(coins, amt)
    print(f"coins={coins}, amount={amt} -> brute={a}, dp={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

Inputs are shaped to force the worst case. (Exponential brute-force versions are shown in the code but omitted from timing where they would blow up — noted per notebook.)

*(The exponential brute force is omitted here; we time the DP as the amount grows.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return ([1, 2, 5], n)   # amount grows -> dp is O(amount * coins)
solutions = {
    "dp O(amount * coins)": coin_change_dp,
}
sizes = [5000, 10000, 20000, 40000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Unbounded DP over a target:** build every amount from smaller amounts; try all choices per state.
- **Greedy fails here:** biggest-coin-first can miss the optimum — DP tries all options.
- **Signal:** "fewest / number of ways to make a target from parts you can reuse".
- **Related problems:** Coin Change II (count ways), Combination Sum, Perfect Squares.
- **Common pitfalls:** (1) greedy assumption; (2) forgetting the impossible (-1) case.